In [ ]:
!pip install -q shap imbalanced-learn xgboost lightgbm joblib

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from scipy.stats import ks_2samp
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, roc_curve
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb
import lightgbm as lgb

sns.set_theme(style="whitegrid")
np.random.seed(42)
print("All libraries loaded successfully!")

# 🏦 Capstone: SafePay Bank — End-to-End Credit Risk & Decision Engine

### 📌 Executive Problem Statement
**SafePay Bank** processes personal credit lines and BNPL (Buy Now, Pay Later) accounts across India. Default accounts ($y = 1$, representing **~23.7%** of borrowers) cause severe capital write-offs (`-₹65,000` per default), while overly strict rejection rules turn away profitable borrowers (`+₹12,000` net interest per good loan).

### 🎯 Project Objectives
1. **Statistical Audit & Feature Engineering:** Diagnose multicollinearity (VIF) and engineer domain ratios (`debt_to_income_ratio`, `delinquency_utilization_idx`).
2. **Data-Leakage-Free ML Pipeline:** Combine `ColumnTransformer` (`StandardScaler` + `OneHotEncoder`) with `SMOTE` and ensemble estimators inside `Pipeline` / `ImbPipeline`.
3. **Model Progression & Benchmarking:** Compare L2 Logistic Regression, Random Forest (SMOTE), Tuned XGBoost, and LightGBM using **PR-AUC (`average_precision`)** and **ROC-AUC**.
4. **Financial Cost-Matrix Threshold Tuning:** Optimize the decision cutoff to maximize **Net Bank Profit**.
5. **Explainable AI (SHAP) & Deployment:** Generate global & borrower-level SHAP explanations and serialize to `.joblib`.

# Step 1 — Realistic FinTech Dataset Generation & Domain Feature Engineering

In [ ]:
def generate_safepay_credit_data(n_samples=5000, random_state=42):
    rng = np.random.default_rng(random_state)
    age = rng.integers(21, 65, n_samples)
    annual_income_lakhs = np.round(rng.lognormal(mean=2.1, sigma=0.55, size=n_samples), 2)
    loan_amount_lakhs = np.round(annual_income_lakhs * rng.uniform(0.15, 0.95, n_samples), 2)
    credit_score = np.clip(rng.normal(710, 65, n_samples), 450, 880).astype(int)
    credit_utilization_pct = np.clip(rng.beta(2.2, 3.5, n_samples) * 100, 2, 99).round(1)
    late_payments_12m = rng.poisson(0.7, n_samples)
    inquiries_6m = rng.poisson(1.1, n_samples)
    employment_type = rng.choice(["Salaried", "Self-Employed", "Freelancer"], size=n_samples, p=[0.62, 0.26, 0.12])
    city_tier = rng.choice(["Tier-1", "Tier-2", "Tier-3"], size=n_samples, p=[0.50, 0.32, 0.18])
    monthly_income_k = np.round((annual_income_lakhs * 100) / 12 + rng.normal(0, 0.5, n_samples), 2)
    dti = loan_amount_lakhs / (annual_income_lakhs + 0.1)
    emp_risk = np.where(employment_type == "Freelancer", 0.55, np.where(employment_type == "Self-Employed", 0.25, 0.0))
    logit = (-3.1 + 1.85 * dti + 0.028 * (credit_utilization_pct - 40) + 0.68 * late_payments_12m + 0.32 * inquiries_6m - 0.012 * (credit_score - 680) + emp_risk + 0.015 * (late_payments_12m * credit_utilization_pct / 10))
    default_flag = (rng.random(n_samples) < (1 / (1 + np.exp(-logit)))).astype(int)
    return pd.DataFrame({"age": age, "annual_income_lakhs": annual_income_lakhs, "monthly_income_k": monthly_income_k, "loan_amount_lakhs": loan_amount_lakhs, "credit_score": credit_score, "credit_utilization_pct": credit_utilization_pct, "late_payments_12m": late_payments_12m, "inquiries_6m": inquiries_6m, "employment_type": employment_type, "city_tier": city_tier, "default": default_flag})

df = generate_safepay_credit_data(n_samples=5000)
df["debt_to_income_ratio"] = (df["loan_amount_lakhs"] / df["annual_income_lakhs"]).round(3)
df["delinquency_utilization_idx"] = ((df["late_payments_12m"] + 1) * (df["credit_utilization_pct"] / 100)).round(3)
print(f"Dataset Shape: {df.shape}")
display(df.head())

# Step 2 — Statistical EDA, Multicollinearity (VIF) Audit & Visuals

In [ ]:
num_cols_raw = ["age", "annual_income_lakhs", "monthly_income_k", "loan_amount_lakhs", "credit_score", "credit_utilization_pct", "late_payments_12m", "inquiries_6m", "debt_to_income_ratio", "delinquency_utilization_idx"]
X_vif = sm.add_constant(df[num_cols_raw])
vif_df = pd.DataFrame({"Feature": num_cols_raw, "VIF": [round(variance_inflation_factor(X_vif.values, i + 1), 2) for i in range(len(num_cols_raw))]}).sort_values("VIF", ascending=False)
display(vif_df.style.hide(axis="index"))
df_clean = df.drop(columns=["monthly_income_k"])

# Step 3 — Leak-Free Preprocessing & Multi-Model Benchmarking

In [ ]:
X = df_clean.drop(columns=["default"])
y = df_clean["default"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = ["employment_type", "city_tier"]
preprocessor = ColumnTransformer([("num", StandardScaler(), numeric_features), ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)])
imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "1. Logistic Regression (L2 + Balanced)": Pipeline([("prep", preprocessor), ("clf", LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000, random_state=42))]),
    "2. Random Forest (SMOTE Pipeline)": ImbPipeline([("prep", preprocessor), ("smote", SMOTE(random_state=42)), ("clf", RandomForestClassifier(n_estimators=250, max_depth=8, random_state=42, n_jobs=-1))]),
    "3. XGBoost (Cost-Sensitive Weighted)": Pipeline([("prep", preprocessor), ("clf", xgb.XGBClassifier(n_estimators=250, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0, scale_pos_weight=imbalance_ratio, eval_metric="logloss", random_state=42))]),
    "4. LightGBM (Cost-Sensitive Weighted)": Pipeline([("prep", preprocessor), ("clf", lgb.LGBMClassifier(n_estimators=250, learning_rate=0.05, num_leaves=15, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=imbalance_ratio, random_state=42, verbose=-1))])
}

leaderboard = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    leaderboard.append({"Model": name, "PR_AUC": round(average_precision_score(y_test, y_prob), 4), "ROC_AUC": round(roc_auc_score(y_test, y_prob), 4), "Recall": round(recall_score(y_test, y_pred), 4), "Precision": round(precision_score(y_test, y_pred), 4), "F1_Score": round(f1_score(y_test, y_pred), 4)})
display(pd.DataFrame(leaderboard).sort_values("PR_AUC", ascending=False).style.hide(axis="index"))

# Step 4 — Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
xgb_pipe = Pipeline([("prep", preprocessor), ("clf", xgb.XGBClassifier(scale_pos_weight=imbalance_ratio, eval_metric="logloss", random_state=42))])
param_grid = {"clf__n_estimators": [150, 250, 350], "clf__learning_rate": [0.02, 0.05, 0.08], "clf__max_depth": [3, 4, 5], "clf__subsample": [0.75, 0.85, 1.0], "clf__colsample_bytree": [0.75, 0.85, 1.0], "clf__reg_alpha": [0.1, 0.5, 1.5], "clf__reg_lambda": [1.0, 3.0, 5.0]}
search = RandomizedSearchCV(xgb_pipe, param_grid, n_iter=15, scoring="average_precision", cv=StratifiedKFold(5, shuffle=True, random_state=42), n_jobs=-1, random_state=42)
search.fit(X_train, y_train)
best_pipeline = search.best_estimator_
y_test_prob = best_pipeline.predict_proba(X_test)[:, 1]
print(f"Test PR-AUC: {average_precision_score(y_test, y_test_prob):.4f} | Test ROC-AUC: {roc_auc_score(y_test, y_test_prob):.4f}")

# Step 5 — Financial Profit-Maximizing Decision Threshold Optimization

In [ ]:
def calculate_net_profit(y_true, y_probs, threshold):
    preds = (y_probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return (tn * 12000) - (fp * 4000) - (fn * 65000), tn, fp, fn, tp

thresholds = np.linspace(0.10, 0.90, 81)
profit_df = pd.DataFrame([{"threshold": t, "net_profit_lakhs": calculate_net_profit(y_test, y_test_prob, t)[0] / 1e5} for t in thresholds])
best_row = profit_df.loc[profit_df["net_profit_lakhs"].idxmax()]
print(f"Profit-Optimal Cutoff: {best_row['threshold']:.2f} | Net Profit: ₹{best_row['net_profit_lakhs']:.2f} Lakhs")

# Step 6 — SHAP Explainability & Save Model Artifact

In [ ]:
prep_step = best_pipeline.named_steps["prep"]
xgb_step = best_pipeline.named_steps["clf"]
all_feature_names = numeric_features + prep_step.named_transformers_["cat"].get_feature_names_out(categorical_features).tolist()
X_test_transformed = pd.DataFrame(prep_step.transform(X_test), columns=all_feature_names)
explainer = shap.TreeExplainer(xgb_step)
shap_values = explainer(X_test_transformed)
shap.summary_plot(shap_values, X_test_transformed)

artifact = {"pipeline": best_pipeline, "optimal_threshold": float(best_row["threshold"]), "feature_columns": X.columns.tolist(), "metrics": {"test_pr_auc": float(average_precision_score(y_test, y_test_prob)), "test_roc_auc": float(roc_auc_score(y_test, y_test_prob))}}
joblib.dump(artifact, "safepay_credit_risk_engine_v1.joblib")
print("Saved safepay_credit_risk_engine_v1.joblib")

# Step 7 — Production Inference Function (Real-Time Loan Scoring)

In [ ]:
def score_loan_applicants(raw_data: pd.DataFrame, artifact_path: str = "safepay_credit_risk_engine_v1.joblib") -> pd.DataFrame:
    processed_df = raw_data.copy()
    bundle = joblib.load(artifact_path)
    pipeline = bundle["pipeline"]
    threshold = bundle["optimal_threshold"]
    processed_df["debt_to_income_ratio"] = (processed_df["loan_amount_lakhs"] / processed_df["annual_income_lakhs"]).round(3)
    processed_df["delinquency_utilization_idx"] = ((processed_df["late_payments_12m"] + 1) * (processed_df["credit_utilization_pct"] / 100)).round(3)
    probabilities = pipeline.predict_proba(processed_df)[:, 1]
    processed_df["default_risk_pct"] = (probabilities * 100).round(2)
    processed_df["credit_decision"] = np.where(probabilities >= threshold, "🚨 REJECT / MANUAL REVIEW", "✅ AUTO-APPROVE")
    return processed_df[["employment_type", "credit_score", "debt_to_income_ratio", "late_payments_12m", "default_risk_pct", "credit_decision"]]

raw_applicants = pd.DataFrame([
    {"age": 34, "annual_income_lakhs": 18.5, "loan_amount_lakhs": 3.0, "credit_score": 795, "credit_utilization_pct": 18.0, "late_payments_12m": 0, "inquiries_6m": 0, "employment_type": "Salaried", "city_tier": "Tier-1"},
    {"age": 26, "annual_income_lakhs": 5.2, "loan_amount_lakhs": 4.5, "credit_score": 560, "credit_utilization_pct": 88.0, "late_payments_12m": 3, "inquiries_6m": 4, "employment_type": "Freelancer", "city_tier": "Tier-3"},
    {"age": 41, "annual_income_lakhs": 11.0, "loan_amount_lakhs": 4.2, "credit_score": 690, "credit_utilization_pct": 52.0, "late_payments_12m": 1, "inquiries_6m": 1, "employment_type": "Self-Employed", "city_tier": "Tier-2"}
])
scored_results = score_loan_applicants(raw_applicants)
active_cutoff = joblib.load("safepay_credit_risk_engine_v1.joblib")["optimal_threshold"]
print(f"=== REAL-TIME LOAN SCORING DECISIONS (Threshold = {active_cutoff:.2f}) ===")
display(scored_results.style.hide(axis="index"))